<div style="
background-color:#EAEAEA;
padding:20px;
border-left:5px solid #6C757D;
border-radius:6px;">

<table style="width:100%; border:none;">
<tr style="border:none;">

<td style="border:none; vertical-align:top;">

<h1 style="font-size:32px; margin-top:0;">
Master's Thesis
</h1>

<hr style="margin:16px 0 22px 0;">

<p style="font-size:22px; line-height:1.5; margin:0;">
<strong>Master's Degree in Advanced Physics</strong> - 
<strong>Universitat de Val?ncia</strong>
</p>

<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">
This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:
</p>

<div style="
font-size:25px;
font-weight:700;
line-height:1.3;
margin-top:14px;
margin-bottom:26px;">
Fast Simulation of Neutrino Oscillations in Matter
</div>

<p style="font-size:14px; line-height:1.55;">
<strong>Author</strong><br>
Juan Ramon Diaz Santos - 
<a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a>
</p>

<p style="font-size:14px; line-height:1.55;">
<strong>Supervisors</strong><br>
Roberto Ruiz de Austri Bazan ?
<a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>
Michele Lucente ?
<a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a>
</p>

<p style="font-size:14px; line-height:1.55; margin-bottom:0;">
<strong>Date</strong><br>
September 2026
</p>

</td>

<td style="
border:none;
width:230px;
padding-left:25px;
text-align:right;
vertical-align:top;">

<img src="../../logo_uv.png"
     alt="Universitat de Val?ncia"
     style="width:200px; margin-top:5px;">

</td>

</tr>
</table>

</div>

# Validation NuSQuIDS 7: BSM (3+1 Sterile) Atmospheric Neutrino Propagation
---

This notebook is the 3+1-sterile counterpart of `nusquids3_atmosphere.ipynb`: it cross-checks tpeanuts's
sterile-extended atmosphere+Earth propagation against nuSQuIDS's `EarthAtm` body, generalized to
`numneu=4` (see `tpeanuts.external.nusquids.core`, extended this session).

**Scope note (NSI):** see `nusquids6_BSM_SolarNeutrino.ipynb` §0 for why an NSI-vs-nuSQuIDS comparison is
not included here (the installed nuSQuIDS 1.13.3 bindings expose no NSI-capable class). This notebook
covers Sterile only, split into two parts:

- **Part A — Null-mixing sanity check:** `sterile_3p1_null_mixing` (all sterile angles zero) must reduce
  exactly to the plain Standard Model in both codes.
- **Part B — Sterile preset comparison:** the same three presets used in `nusquids6...ipynb` and
  `validation_intrinsic2_BSM_perturvative_vs_numerical.ipynb` §7, for an initial $\nu_\mu$ (the channel
  atmospheric sterile searches such as IceCube target).

Both parts cover a downward-going angle (pure atmosphere, no Earth crossing) and an up-going angle
(atmosphere + Earth crossing), following `nusquids3_atmosphere.ipynb`'s $\cos\theta_z$ convention.

**Environment note:** requires the nuSQuIDS Python bindings importable in-process alongside tpeanuts —
see `nusquids6...ipynb` §0 for the WSL/`corsika8-venv` environment this project uses for that. Degrades to
NaN placeholders, without crashing, if nuSQuIDS is not importable in the active kernel.

## Table of Contents

| # | Section |
|---|---------|
| [0](#0.-Theory-Background) | **Theory Background**: EarthAtm cross-check methodology |
| [1](#1.-Libraries) | **Libraries** |
| [2](#2.-Paths-and-Configuration) | **Paths and Configuration** |
| [3](#3.-Part-A----Null-Mixing-Sanity-Check) | **Part A — Null-Mixing Sanity Check** |
| [4](#4.-Part-B----Sterile-Preset-Comparison) | **Part B — Sterile Preset Comparison** |
| [5](#5.-Summary) | **Summary** |

## 0. Theory Background

### 0.1 `atmosphere_evolutor` is production-to-surface *only*

`tpeanuts.medium.atmosphere.evolutor.atmosphere_evolutor` implements exactly what its own module
docstring states: "production height h -> earth surface". It is **not**, by itself, a full
production-to-detector chain for a genuinely up-going trajectory: an early version of this notebook called
it (via `atmosphere_probability_transition`) directly for both down-going and up-going angles and found that
`atmosphere_probability_transition`'s output was numerically identical for $\theta_z$ and $180^\circ-\theta_z$ even
though the two geometries are completely different (no Earth crossing versus a full Earth-diameter
crossing) -- the underground path length *is* computed correctly by
`tpeanuts.medium.atmosphere.geometry.underground_path_length` for the up-going case, but
`atmosphere_evolutor` never folds that leg into its returned operator. This is not a bug: composing the
Earth-crossing leg is the responsibility of a **separate** `earth_probability_state` call, exactly as
`validation_intrinsic2_BSM_perturvative_vs_numerical.ipynb`'s `atmosphere_to_detector` helper already does,
and exactly why `nusquids3_atmosphere.ipynb` deliberately restricts itself to $\cos\theta_z\in[0,1]$
(down-going only, where the missing leg is trivially absent). This notebook needs the up-going regime (see
Section 0.2), so it reuses that same two-step composition:

```python
S_atm, _ = atmosphere_evolutor(oscillation, E_MeV, h_km, theta_deg, depth_km, atmosphere=..., method="analytical")
surface_state = S_atm @ initial_flavour_state
eta = pi - radians(theta_deg)
P = earth_probability_state_analytical(surface_state, earth_profile, oscillation, E_MeV, eta, depth_km, massbasis=False)
```

`earth_probability_state`'s own "above horizon" identity behaviour ($\eta\ge\pi/2$, i.e. $\theta_z\le90^\circ$) means this
composition is automatically correct for down-going angles too (the Earth leg reduces to the identity), so
a single helper covers both regimes without special-casing.

### 0.2 Same-density-model comparison, and where it breaks down

Like `nusquids3...ipynb`, the *atmosphere* leg uses `atmosphere_density_source="nusquids_earthatm"` in both
codes, isolating pure numerical-integration residuals from atmosphere density-model differences for the
down-going point. The up-going point's *Earth* leg, however, necessarily uses tpeanuts's own PREM fit
(`config.earth_density_file`) on the tpeanuts side and nuSQuIDS's own bundled PREM table inside its
`EarthAtm` body -- the same already-characterized residual source as
`nusquids2_solar.ipynb`/`nusquids5.../nusquids6...ipynb`'s Earth comparisons, now compounded with using
tpeanuts's *analytical* (perturbative) evolutor rather than its numerical one (see Section 0.3).

### 0.3 nuSQuIDS EarthAtm generalized to numneu=4

`EarthAtm.MakeTrackWithCosine(cos_zenith)` and the underlying `nuSQUIDS(numneu, ...)` solver both already
support `numneu>=4` natively (documented as "4+ for sterile" in the installed 1.13.3 bindings). The tpeanuts
side of the comparison uses `method="analytical"` for both `atmosphere_evolutor` and `earth_probability_state_analytical`
(matching `nusquids5.../nusquids6...ipynb`'s choice to validate the perturbative branch against nuSQuIDS's
independent, numerically-exact ODE reference, rather than tpeanuts's own numerical branch against itself).

**Expected results:** down-going agreement at the sub-ppb level, matching `nusquids3...ipynb`'s own
Standard Model comparison; up-going agreement at the percent level (Earth density-model mismatch, Section
0.2), still much smaller than the genuine sterile signal size explored in Part B.

### References

- Argüelles, C. A., Salvado, J. & Weaver, C. N. (2022). *nuSQuIDS: A toolbox for neutrino propagation*.
  Comput. Phys. Commun. 277, 108346.
- C. Giunti, M. Marrone and A. Palazzo, *Combined 3+1 global analysis*, arXiv:1612.01087 (2017).

## 1. Libraries

In [ ]:
from __future__ import annotations

%matplotlib inline
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from tpeanuts.notebooks.notebookConfig import load_notebook_config
from tpeanuts.notebooks.notebooks_helper import (
    to_numpy, save_and_show, compare_probability_grids, nusquids_precision_summary,
)
from tpeanuts.external.nusquids.core import (
    NuSQuIDSConfig, is_available as nusquids_is_available, probability_atmosphere,
)
from tpeanuts.util.context import RuntimeContext
from tpeanuts.core.common.oscillation import OscillationParameters
from tpeanuts.config.propagation import PropagationConfig
from tpeanuts.medium.atmosphere.evolutor import atmosphere_evolutor
from tpeanuts.medium.atmosphere.profile import AtmosphereParameters
from tpeanuts.medium.earth.profile import EarthParameters, EarthProfile
from tpeanuts.medium.earth.probability import earth_probability_state_analytical

## 2. Paths and Configuration

**Expected results:** the package/output directories exist, and `nusquids_is_available()` is True when
run under the WSL nuSQuIDS+tpeanuts kernel described in §0.

In [ ]:
config = load_notebook_config()
ctx = RuntimeContext.resolve(config.device, config.dtype)
CDTYPE = torch.complex128 if ctx.dtype == torch.float64 else torch.complex64
OUTPUT_DIR = config.output_dir("validation", "nusquids")
SHOW = config.show_plots

NSQ_AVAILABLE = nusquids_is_available()

sterile_null_oscillation = PropagationConfig.oscillation_parameters_from_preset("sterile_3p1_null_mixing", antinu=False, context=ctx)
STERILE_PRESETS = [
    "sterile_3p1_bestfit_giunti2017",
    "sterile_3p1_benchmark_icecube",
    "sterile_3p1_two_flavour_limit",
]

# Same atmospheric density model in both codes for the atmosphere leg (see Section 0.2); matches nusquids3.
atm_params = AtmosphereParameters(atmosphere_density_source="nusquids_earthatm", nsteps=600, matter=True)
earth = EarthProfile(
    params=EarthParameters(profile_perturbative_kwargs={"density_file": str(config.earth_density_file), "tabulated_density": False}),
    context=ctx,
)
H_PROD_KM = 22.0
DEPTH_KM = 0.0

E_CHECK_GEV = [0.5, 3.0, 10.0]
COS_ZENITH_CHECK = [0.5, -0.5]   # down-going (atmosphere only) and up-going (+ Earth crossing), see Section 0.1-0.2
INIT_FLAVOUR = 1  # nu_mu

print(f"Device : {ctx.device}   dtype : {ctx.dtype}")
print(f"Output : {OUTPUT_DIR}")
print(f"nuSQuIDS available: {NSQ_AVAILABLE}")


def nsq_config_from_oscillation(oscillation, *, numneu=3):
    """Build a NuSQuIDSConfig sharing an OscillationParameters' angles/splittings (see nusquids3/6)."""
    params = oscillation.pmns.params
    kwargs = dict(
        theta12=float(params.theta12), theta13=float(params.theta13), theta23=float(params.theta23),
        delta_cp=float(params.delta), DeltamSq21=float(oscillation.mass_spectrum.DeltamSq21), DeltamSq3l=float(oscillation.mass_spectrum.DeltamSq3l),
        numneu=numneu,
    )
    if numneu >= 4:
        sp = oscillation.pmns.sterile_params
        kwargs.update(
            theta14=float(sp.theta14), theta24=float(sp.theta24), theta34=float(sp.theta34),
            delta14=float(sp.delta14), delta24=float(sp.delta24), DeltamSq41=float(oscillation.mass_spectrum.DeltamSq41),
        )
    return NuSQuIDSConfig(**kwargs)


def tpeanuts_atm_prob(oscillation, *, E_GeV, theta_deg, numneu):
    """P(nu_mu -> all) from tpeanuts's two-step atmosphere+Earth chain (see Section 0.1)."""
    E_MeV_t = torch.tensor(E_GeV * 1.0e3, dtype=ctx.dtype, device=ctx.device)
    theta_t = torch.tensor(theta_deg, dtype=ctx.dtype, device=ctx.device)
    S_atm, _ = atmosphere_evolutor(
        oscillation, E_MeV_t, H_PROD_KM, theta_t, DEPTH_KM, atmosphere=atm_params, context=ctx, method="analytical",
    )
    state = torch.eye(numneu, dtype=CDTYPE, device=ctx.device)[INIT_FLAVOUR]
    surface_state = S_atm @ state
    eta_t = torch.tensor(math.pi - math.radians(theta_deg), dtype=ctx.dtype, device=ctx.device)
    P = earth_probability_state_analytical(surface_state, earth, oscillation, E_MeV_t, eta_t, DEPTH_KM, massbasis=False)
    return to_numpy(P)


def nusquids_atm_prob(*, E_GeV, cos_zenith, numneu, config):
    if not NSQ_AVAILABLE:
        return np.full(numneu, float("nan"))
    return probability_atmosphere(E_GeV=E_GeV, cos_zenith=cos_zenith, initial_flavour=INIT_FLAVOUR, config=config)


def compare_atmosphere(oscillation, *, numneu, label):
    """Run the tpeanuts-vs-nuSQuIDS EarthAtm comparison over the (E_GeV, cos_zenith) check points."""
    nsq_cfg = nsq_config_from_oscillation(oscillation, numneu=numneu)
    prob_cols = [f"P_{i}" for i in range(numneu)]
    tp_rows, nsq_rows = [], []
    for cz in COS_ZENITH_CHECK:
        theta_deg = math.degrees(math.acos(float(np.clip(cz, -1.0, 1.0))))
        for E_GeV in E_CHECK_GEV:
            tp = tpeanuts_atm_prob(oscillation, E_GeV=E_GeV, theta_deg=theta_deg, numneu=numneu)
            row_tp = {"cos_zenith": cz, "E_GeV": E_GeV}
            row_tp.update({c: float(tp[i]) for i, c in enumerate(prob_cols)})
            tp_rows.append(row_tp)

            try:
                nsq = nusquids_atm_prob(E_GeV=E_GeV, cos_zenith=cz, numneu=numneu, config=nsq_cfg)
            except Exception as exc:  # pragma: no cover - environment dependent
                print(f"  [skipped] {label} E={E_GeV} cos_zenith={cz}: {exc}")
                nsq = np.full(numneu, float("nan"))
            row_nsq = {"cos_zenith": cz, "E_GeV": E_GeV}
            row_nsq.update({c: float(nsq[i]) for i, c in enumerate(prob_cols)})
            nsq_rows.append(row_nsq)

    tp_df = pd.DataFrame(tp_rows)
    nsq_df = pd.DataFrame(nsq_rows)
    comparison = compare_probability_grids(tp_df, nsq_df, ["cos_zenith", "E_GeV"], prob_cols=prob_cols)
    summary = nusquids_precision_summary(comparison)
    print(f"--- {label} ---")
    print(summary.to_string())
    return comparison, summary

## 3. Part A — Null-Mixing Sanity Check

**Expected results:** agreement at the same order as `nusquids3_atmosphere.ipynb`'s own Standard Model
comparison for the down-going point (sub-ppb); a looser but still small (percent-level) residual for the
up-going point (Earth density-model difference, see Section 0.2). `P_3` (sterile) stays at floating-point
zero in both codes.

In [ ]:
null_comparison, null_summary = compare_atmosphere(
    sterile_null_oscillation, numneu=4, label="Null-mixing (sterile_3p1_null_mixing)",
)
if not null_comparison.empty:
    print()
    print("Sterile-flavour probability (P_3), tpeanuts:", null_comparison["P_3_tp"].abs().max())
    if "P_3_nsq" in null_comparison.columns:
        print("Sterile-flavour probability (P_3), nuSQuIDS:", null_comparison["P_3_nsq"].abs().max())

## 4. Part B — Sterile Preset Comparison

**Runtime note:** as in `nusquids6_BSM_SolarNeutrino.ipynb` §4, these presets' non-zero `DeltamSq41`
(0.3-1.7 eV$^2$) combined with `NuSQuIDSConfig`'s tight default ODE tolerances make nuSQuIDS's adaptive
integrator take several minutes per preset (confirmed by direct timing during development for the
solar/Earth case; the same physical cause -- a short sterile oscillation length forcing many small
integrator steps -- applies here). Reduce `E_CHECK_GEV`/`COS_ZENITH_CHECK` or loosen `rel_error`/
`abs_error` in `nsq_config_from_oscillation` first if a quicker run is needed.

**Expected results:** agreement at the same order as Part A; the up-going, long-baseline point should show
a non-trivial sterile-flavour probability, most visibly for `sterile_3p1_benchmark_icecube` (largest
$\theta_{24}$ tuned for exactly this $\nu_\mu$ atmospheric channel).

In [ ]:
preset_results = {}
for preset_name in STERILE_PRESETS:
    oscillation = PropagationConfig.oscillation_parameters_from_preset(preset_name, context=ctx, antinu=False)
    comparison, summary = compare_atmosphere(oscillation, numneu=4, label=preset_name)
    preset_results[preset_name] = (comparison, summary)
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
labels = ["Null-mixing"] + STERILE_PRESETS
max_rel = [null_summary.loc["max_rel_err", "value"]] + [preset_results[p][1].loc["max_rel_err", "value"] for p in STERILE_PRESETS]
max_abs = [null_summary.loc["max_abs_err", "value"]] + [preset_results[p][1].loc["max_abs_err", "value"] for p in STERILE_PRESETS]
axes[0].bar(range(len(labels)), max_abs, color="C0"); axes[0].set_title("Maximum absolute error"); axes[0].set_yscale("log")
axes[1].bar(range(len(labels)), max_rel, color="C1"); axes[1].set_title("Maximum relative error"); axes[1].set_yscale("log")
for ax in axes:
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
    ax.grid(alpha=0.25, axis="y")
fig.suptitle("tpeanuts vs nuSQuIDS: atmosphere-to-detector (nu_mu), 3+1 sterile")
fig.tight_layout()
save_and_show("vn7_fig1_sterile_atm_bar.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW)

## 5. Summary

In [ ]:
all_rows = [{"case": "Null-mixing (sterile_3p1_null_mixing)", **null_summary["value"].to_dict()}]
for preset_name in STERILE_PRESETS:
    all_rows.append({"case": preset_name, **preset_results[preset_name][1]["value"].to_dict()})
summary_table = pd.DataFrame(all_rows).set_index("case")
display(summary_table)
summary_table.to_csv(OUTPUT_DIR / "vn7_summary.csv")

null_comparison.to_csv(OUTPUT_DIR / "vn7_null_mixing_comparison.csv", index=False)
for preset_name in STERILE_PRESETS:
    preset_results[preset_name][0].to_csv(OUTPUT_DIR / f"vn7_{preset_name}_comparison.csv", index=False)

print("Saved:", OUTPUT_DIR / "vn7_summary.csv")
if NSQ_AVAILABLE:
    print(f"All {len(summary_table)} tpeanuts-vs-nuSQuIDS sterile atmosphere comparisons completed.")
else:
    print("nuSQuIDS was not available in this kernel: comparisons are NaN placeholders (see Section 0 environment note).")